In [1]:
import itertools
import time
import numpy as np
import scipy
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
from sklearn import utils, neighbors
from tqdm import tqdm

import ripserplusplus as rpp_py
from RTD_Lite import RTD_Lite_summ_only

In [41]:
batch_size_train = 256
learning_rate = 1e-4
hidden_dimension = 16
lam_bda = 1.
n_epochs = 250

random_seed = 1
torch.manual_seed(random_seed)

#### Read data

In [3]:
import idx2numpy

imagefile = 'FMNIST/train-images-idx3-ubyte'
imagearray = idx2numpy.convert_from_file(imagefile)

imagearray = imagearray.reshape(60000, 28*28).astype(np.double) / 255.0

int_train = utils.shuffle(imagearray)

#### Model architecture

In [42]:
HDD_DIM = 512

class Autoencoder(nn.Module):  # [128*128 - 512 - 512 - med]
  def __init__(self, input_dim):
    super(Autoencoder, self).__init__()  
    self.layer1 = nn.Linear(input_dim, HDD_DIM)  # no labels
    self.layer2 = nn.Linear(HDD_DIM , HDD_DIM // 2)
    self.layer3 = nn.Linear(HDD_DIM // 2, HDD_DIM // 4)
    self.layer6 = nn.Linear(HDD_DIM // 4, hidden_dimension)
    
    self.layer7 = nn.Linear(hidden_dimension, HDD_DIM // 4)
    self.layer8 = nn.Linear(HDD_DIM // 4, HDD_DIM // 2)
    self.layer10 = nn.Linear(HDD_DIM // 2, HDD_DIM)
    self.layer12 = nn.Linear(HDD_DIM , input_dim)
    
    self.input_dim = input_dim
    
    # This parameter governs the proportion betweeen normalized distances in dormant and latent spaces
    self.quant_inner_dimension = torch.nn.Parameter(data=torch.ones(1), requires_grad=True)
    
    
  def encode(self, x):             
    z = nn.ReLU()(self.layer1(x.view(-1, self.input_dim)))
    z = nn.ReLU()(self.layer2(z))
    z = nn.ReLU()(self.layer3(z))
    z = self.layer6(z)
    return z 

  def decode(self, x):              
    z = nn.ReLU()(self.layer7(x))
    z = nn.ReLU()(self.layer8(z))
    z = nn.ReLU()(self.layer10(z))
    z = self.layer12(z)
    return z

  def forward(self, x):
    z = self.encode(x)
    oupt = self.decode(z)
    return oupt

### Get quantile

In [43]:
dsts = scipy.spatial.distance.pdist(int_train)
quant = np.quantile(dsts, 0.9)

In [44]:
ae = Autoencoder(int_train.shape[1]).to('cuda')
ae.train()
losses = []
loss_func = torch.nn.MSELoss()
opt = torch.optim.Adam(ae.parameters(), betas=(0.9, 0.999), lr=learning_rate, weight_decay=1e-5)

#### Training

In [45]:
print("Starting training")
np.random.seed(42)

start = time.time()
for epoch in range(n_epochs):
    ae.train()
    running_loss = 0.0    
    int_train = utils.shuffle(int_train)

    for b_idx in range((int_train.shape[0] + batch_size_train - 1) // batch_size_train):
        batch = torch.tensor(int_train[b_idx * batch_size_train : (b_idx + 1) * batch_size_train ], requires_grad=False).float()
        X = batch.to('cuda')
        mid = ae.encode(X).float()
        oupt = ae.decode(mid)
        loss_val = loss_func(oupt, X)
        if epoch >= 0:
            rtd_lite_loss = RTD_Lite_summ_only(X, mid, quant, ae.quant_inner_dimension)() / batch.shape[0]
            loss_val = loss_val + rtd_lite_loss * lapp_dumb
        
        running_loss += loss_val.item()
        losses.append(loss_val.item())
        
        opt.zero_grad()
        loss_val.backward() 
        opt.step()

    if epoch != 0:

        print("epoch = %6d" % epoch, end="")
        print("  curr batch loss = ", running_loss / (1.0 + b_idx), end="")
    print("")
    
print("time_elapsed: ", time.time() - start)
print("Training complete ")

Starting training

epoch =      1  curr batch loss =  0.0387569139136913
epoch =      2  curr batch loss =  0.03410107486267039
epoch =      3  curr batch loss =  0.03259159894065654
epoch =      4  curr batch loss =  0.03161612724528668
epoch =      5  curr batch loss =  0.030897191904009656
epoch =      6  curr batch loss =  0.030289016378686782
epoch =      7  curr batch loss =  0.029869650692698802
epoch =      8  curr batch loss =  0.029512039564074354
epoch =      9  curr batch loss =  0.02929779054953697
epoch =     10  curr batch loss =  0.029061129102681545
epoch =     11  curr batch loss =  0.028835692700553448
epoch =     12  curr batch loss =  0.02866543879375813
epoch =     13  curr batch loss =  0.028509507502647156
epoch =     14  curr batch loss =  0.028317144600317833
epoch =     15  curr batch loss =  0.028188940795495155
epoch =     16  curr batch loss =  0.02800374537547852
epoch =     17  curr batch loss =  0.027876237288434455
epoch =     18  curr batch loss =  0.

#### Calculate distance linear correlation 

In [46]:
import scipy

embedding = ae.encode(torch.tensor(int_train[:20000], requires_grad=False).float().to('cuda')).cpu().detach().numpy()
    
r_large = scipy.spatial.distance.pdist(int_train[:20000]) 
r_smoll = scipy.spatial.distance.pdist(embedding[:20000]) 
print('l.c. = ', np.corrcoef(r_smoll, r_large)[0][1])

l.c. =  0.7009217992927571


### Further metrics

In [47]:
dsts = scipy.spatial.distance.pdist(embedding)
quant_inner_dimension = np.quantile(dsts, 0.9)

#### W.D. H_0 and RTD

In [49]:
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PairwiseDistance

def get_Wass_distances(initial_data, embeddings):
    ripser_model = VietorisRipsPersistence(metric='precomputed', homology_dimensions=[0], collapse_edges=False)
    wass_model = PairwiseDistance(metric='wasserstein', metric_params={'p': 1.0}, order=None, n_jobs=None)

    wass_dists = []
    
    batch_size_eval = 2048
    for b_idx in tqdm(range((initial_data.shape[0] + batch_size_eval - 1) // batch_size_eval)):
        batch = [torch.tensor(initial_data[b_idx * batch_size_eval : (b_idx + 1) * batch_size_eval], requires_grad=False).float()]    
        X = batch[0].to('cuda')
    
        mid = torch.tensor(embeddings[b_idx * len(X) : (b_idx + 1) * len(X)], requires_grad=False)

        X = torch.cdist(X, X)
        mid = torch.cdist(mid, mid)
        X /= quant #torch.quantile(X, 0.9)
        mid /= quant_inner_dimension # torch.quantile(mid, 0.9)
        X_dgms = ripser_model.fit_transform(X.reshape(1, *X.shape).cpu().detach())
        mid_dgms = ripser_model.fit_transform(mid.reshape(1, *mid.shape).cpu().detach())
        wass_model.fit(X_dgms)

        wass_dists.append(wass_model.transform(mid_dgms)[0][0])

    wass_dists = np.vstack(wass_dists)
    print('Wasserstein distances [0, 1]: ', np.mean(wass_dists, 0), '+-', np.std(wass_dists, 0))
    return wass_dists
    
def get_final_RTD(initial_data, embeddings):
    rtds = []
    batch_size_eval = 80
    for b_idx in tqdm(range(initial_data.shape[0] // batch_size_eval)):
        batch = [torch.tensor(initial_data[b_idx * batch_size_eval : (b_idx + 1) * batch_size_eval], requires_grad=False).float()]    
        X = batch[0]#.to('cuda')
    
        mid = torch.tensor(embeddings[b_idx * len(X) : (b_idx + 1) * len(X)], requires_grad=False)#.to('cuda')

        X = torch.cdist(X, X)
        mid = torch.cdist(mid, mid)
        X /= quant #torch.quantile(X, 0.9)                   # 0.9 quantile for initial data
        mid /= quant_inner_dimension # torch.quantile(mid, 0.9) # 0.9 quantile for embeddings distances
        
        Dzz = torch.zeros(X.shape)#.to('cuda')
        Dr12 = torch.minimum(X, mid)
        
        DX = torch.cat((torch.cat((Dzz, X), 1), torch.cat((X, Dr12), 1)), 0).detach().numpy()
        DX_2 = torch.cat((torch.cat((Dzz, mid), 1), torch.cat((mid, Dr12), 1)), 0).detach().numpy()
        
        DX = (DX + DX.T) / 2.0
        DX_2 = (DX_2 + DX_2.T) / 2.0
        
        DX -= np.diag(np.diag(DX))
        DX_2 -= np.diag(np.diag(DX_2))
        
        dgm = rpp_py.run("--format distance --mode rtd --dim 1", DX)['dgms'][1]
        val = np.sum([interval[1] - interval[0] for interval in dgm])
        dgm2 = rpp_py.run("--format distance --mode rtd --dim 1", DX_2)['dgms'][1]
        val2 = np.sum([interval[1] - interval[0] for interval in dgm2])

        rtds.append(0.5 * (val + val2))

    rtds = np.vstack(rtds)
  #  print(rtds)
    print('RTD final score: ', np.mean(rtds), '+-', np.std(rtds), 'Range :', np.min(rtds), np.max(rtds))

In [50]:
with torch.no_grad():
    all_xy = ae.encode(torch.tensor(int_train, requires_grad=False).float().to('cuda')).cpu().detach().numpy()

In [ ]:
#int_train, lab_train, embeddings = utils.shuffle(int_train, lab_train, embeddings, random_state=1337)

wass_dists = get_Wass_distances(int_train, all_xy)
get_final_RTD(int_train, all_xy)

#### Triplet Accuracy

In [1]:
from itertools import combinations, combinations_with_replacement, product

def zero_out_diagonal(distances):# make 0 on diagonal
    return distances * (np.ones_like(distances) - np.eye(*distances.shape))

def pdist_gpu(a, b, device = 'cuda:0'):
    A = torch.tensor(a, dtype = torch.float64)
    B = torch.tensor(b, dtype = torch.float64)

    size = (A.shape[0] + B.shape[0]) * A.shape[1] / 1e9
    max_size = 0.2

    if size > max_size:
        parts = int(size / max_size) + 1
    else:
        parts = 1

    pdist = np.zeros((A.shape[0], B.shape[0]))
    At = A.to(device)

    for p in range(parts):
        i1 = int(p * B.shape[0] / parts)
        i2 = int((p + 1) * B.shape[0] / parts)
        i2 = min(i2, B.shape[0])

        Bt = B[i1:i2].to(device)
        pt = torch.cdist(At, Bt)
        pdist[:, i1:i2] = pt.cpu()

        del Bt, pt
        torch.cuda.empty_cache()

    del At

    return pdist

def triplet_accuracy(input_data, latent_data, triplets=None):
    # calculate distance matricies
    input_data = input_data.reshape(input_data.shape[0], -1)
    input_distances = zero_out_diagonal(pdist_gpu(input_data, input_data))
    latent_data = latent_data.reshape(latent_data.shape[0], -1)
    latent_distances = zero_out_diagonal(pdist_gpu(latent_data, latent_data))
    # generate triplets
    if triplets is None:
        triplets = np.asarray(list(combinations(range(len(input_data)), r=3)))
    i_s = triplets[:, 0]
    j_s = triplets[:, 1]
    k_s = triplets[:, 2]
    acc = (np.logical_xor(
        input_distances[i_s, j_s] < input_distances[i_s, k_s], 
        latent_distances[i_s, j_s] < latent_distances[i_s, k_s]
    ) == False)
    acc = np.mean(acc.astype(np.int32))
    return acc


def avg_triplet_accuracy(input_data, latent_data, batch_size=128, n_runs=20):
    # average over batches
    accs = []
    triplets = np.asarray(list(combinations(range(min(batch_size, len(input_data))), r=3)))
    if batch_size > len(input_data):
        accs.append(triplet_accuracy(input_data, latent_data, triplets=triplets))
        return accs
    for _ in range(n_runs):
        ids = np.random.choice(np.arange(len(input_data)), size=batch_size, replace=False)
        accs.append(triplet_accuracy(input_data[ids], latent_data[ids], triplets=triplets))
    return accs

In [ ]:
triplets = avg_triplet_accuracy(int_train, all_xy)
print(np.mean(triplets), '+-', np.std(triplets))